# Tutorial: Palette catalog summaries

Audience:
- Analysts and developers who want a compact summary of a color catalog.

Prerequisites:
- Basic Python functions, dictionaries, and lists.
- Familiarity with CSV files.

Learning goals:
- Read a UTF-8 CSV with headers using only the Python standard library.
- Normalize and group palette rows by primary color.
- Produce a stable sorted summary for downstream use.


## Outline

1. Implement `summarize_palettes(csv_path)`.
2. Run quick checks for duplicates, reordering, lowercase colors, and quoted commas.
3. Build `palette_summary` from `/inputs/colors.csv`.
4. Review exercises.


In [ ]:
from __future__ import annotations

import csv
from collections import defaultdict


def summarize_palettes(csv_path):
    counts = defaultdict(int)
    product_type_sets = defaultdict(set)

    with open(csv_path, "r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        if reader.fieldnames is None:
            return []

        for row in reader:
            primary = (row.get("Primary (Hex)") or "").strip().upper()
            product_type = (row.get("Product Type") or "").strip()

            counts[primary] += 1
            product_type_sets[primary].add(product_type)

    summary = [
        {
            "primary": primary,
            "count": counts[primary],
            "product_types": sorted(product_type_sets[primary]),
        }
        for primary in counts
    ]

    summary.sort(key=lambda item: (-item["count"], item["primary"]))
    return summary


## Step 1 - Quick robustness checks

The next cell builds a tiny temporary catalog to verify the function handles:
- schema-compatible reordered rows
- duplicate rows
- lowercase primary colors
- quoted product names that contain commas
- an empty catalog with only headers


In [ ]:
import os
import tempfile

sample_rows = [
    {"No": "2", "Product Type": "  Alpha  ", "Keywords": "k", "Primary (Hex)": " #aa00aa ", "Secondary (Hex)": "", "CTA (Hex)": "", "Background (Hex)": "", "Text (Hex)": "", "Border (Hex)": "", "Notes": ""},
    {"No": "1", "Product Type": "Alpha, Pro", "Keywords": "k", "Primary (Hex)": "#AA00AA", "Secondary (Hex)": "", "CTA (Hex)": "", "Background (Hex)": "", "Text (Hex)": "", "Border (Hex)": "", "Notes": ""},
    {"No": "3", "Product Type": "Beta", "Keywords": "k", "Primary (Hex)": "#00ff00", "Secondary (Hex)": "", "CTA (Hex)": "", "Background (Hex)": "", "Text (Hex)": "", "Border (Hex)": "", "Notes": ""},
    {"No": "4", "Product Type": "Beta", "Keywords": "k", "Primary (Hex)": "#00FF00", "Secondary (Hex)": "", "CTA (Hex)": "", "Background (Hex)": "", "Text (Hex)": "", "Border (Hex)": "", "Notes": ""},
]

fieldnames = ["No", "Product Type", "Keywords", "Primary (Hex)", "Secondary (Hex)", "CTA (Hex)", "Background (Hex)", "Text (Hex)", "Border (Hex)", "Notes"]

with tempfile.TemporaryDirectory() as tmpdir:
    sample_path = os.path.join(tmpdir, "sample.csv")
    with open(sample_path, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(sample_rows)

    check_summary = summarize_palettes(sample_path)
    assert check_summary[0]["primary"] == "#00FF00"
    assert check_summary[0]["count"] == 2
    assert check_summary[0]["product_types"] == ["Beta"]
    assert check_summary[1]["primary"] == "#AA00AA"
    assert check_summary[1]["count"] == 2
    assert check_summary[1]["product_types"] == ["Alpha", "Alpha, Pro"]

    empty_path = os.path.join(tmpdir, "empty.csv")
    with open(empty_path, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()

    assert summarize_palettes(empty_path) == []


In [ ]:
palette_summary = summarize_palettes("/inputs/colors.csv")
palette_summary[:5]


## Exercises

- Add a filter that keeps only groups with `count >= 2`.
- Extend each summary object with the first seen row number while preserving current fields.
- Write a test CSV where `Product Type` is blank and decide how to handle it.
